In [1]:
import pymatching
import sinter
from typing import List
import os
import numpy as np
import stim
import matplotlib.pyplot as plt
print(stim.__version__,os.cpu_count())

1.15.0 12


In [ ]:
circuit = stim.Circuit()
circuit.append("H", [0])
circuit.append("CNOT",[0,1])
circuit.append("M",[0,1])

circuit
circuit.diagram()

In [ ]:
sampler =circuit.compile_sampler()
print(sampler.sample(shots=10))

In [ ]:
circuit.append("DETECTOR",[stim.target_rec(-1),stim.target_rec(-2)])
circuit.diagram()

In [ ]:
sampler=circuit.compile_detector_sampler()
print(sampler.sample(shots=10))

In [ ]:
circuit=stim.Circuit("""
    H 0
    TICK

    CX 0 1
    X_ERROR(0.2) 0 1
    TICK

    M 0 1
    DETECTOR rec[-1] rec[-2]
""")

circuit

In [ ]:
circuit.diagram("timeline-svg")

In [ ]:
circuit.diagram("timeslice-svg")

In [ ]:
sampler=circuit.compile_detector_sampler()
print(sampler.sample(shots=10))

In [2]:
circuit=stim.Circuit.generated(
    "repetition_code:memory",
    rounds=25,
    distance=9,
    before_round_data_depolarization=0.04,
    before_measure_flip_probability=0.01
)

print(repr(circuit))

stim.Circuit('''
    R 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16
    TICK
    DEPOLARIZE1(0.04) 0 2 4 6 8 10 12 14 16
    CX 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
    TICK
    CX 2 1 4 3 6 5 8 7 10 9 12 11 14 13 16 15
    TICK
    X_ERROR(0.01) 1 3 5 7 9 11 13 15
    MR 1 3 5 7 9 11 13 15
    DETECTOR(1, 0) rec[-8]
    DETECTOR(3, 0) rec[-7]
    DETECTOR(5, 0) rec[-6]
    DETECTOR(7, 0) rec[-5]
    DETECTOR(9, 0) rec[-4]
    DETECTOR(11, 0) rec[-3]
    DETECTOR(13, 0) rec[-2]
    DETECTOR(15, 0) rec[-1]
    REPEAT 24 {
        TICK
        DEPOLARIZE1(0.04) 0 2 4 6 8 10 12 14 16
        CX 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
        TICK
        CX 2 1 4 3 6 5 8 7 10 9 12 11 14 13 16 15
        TICK
        X_ERROR(0.01) 1 3 5 7 9 11 13 15
        MR 1 3 5 7 9 11 13 15
        SHIFT_COORDS(0, 1)
        DETECTOR(1, 0) rec[-8] rec[-16]
        DETECTOR(3, 0) rec[-7] rec[-15]
        DETECTOR(5, 0) rec[-6] rec[-14]
        DETECTOR(7, 0) rec[-5] rec[-13]
        DETECTOR(9, 0) rec[-4] r

In [ ]:
sampler=circuit.compile_sampler()
one_sample=sampler.sample(shots=1)[0]
for k in range(0,len(one_sample),8):
    timeslice=one_sample[k:k+8]
    print("".join("1" if e else "_" for e in timeslice))

In [ ]:
dem = circuit.detector_error_model()
print(repr(dem))

In [3]:
def count_logical_errors(circuit: stim.Circuit, num_shots: int) -> int:
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(num_shots,separate_observables=True)
    print(detection_events)
    print(observable_flips)

    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    predictions = matcher.decode_batch(detection_events)

    num_errors = 0
    for shot in range(num_shots):
        actual_for_shot = observable_flips[shot]
        predictions_for_shot = predictions[shot]
        if not np.array_equal(actual_for_shot,predictions_for_shot):
            num_errors += 1
    return num_errors

In [5]:
num_shots = 1
num_logical_errors = count_logical_errors(circuit,num_shots)
print("there were", num_logical_errors, "wrong predictions (logical errors) out of", num_shots, "shots")

[[False False False False False  True  True False False False False False
  False False  True False False False False False False False  True False
  False False False False False False False False False  True  True False
  False False False False False False False False False False False False
  False False False False False False False False False False False False
  False False False False False False False False False False False False
  False False False False False False False False False False False False
  False  True  True False False False False False False False False False
  False False False False False False False False  True False False False
  False False False False False False False False False False False False
  False False False False False False False False False False False False
  False False False False False False False False False False False False
  False False False False False False False False False  True  True False
  False False False False False False 

In [ ]:
num_shots = 10_000
for d in [3,5,7]:
    xs = []
    ys = []

    for noise in [0.1,0.2,0.3,0.4,0.5]:
        circuit = stim.Circuit.generated(
            "repetition_code:memory",
            rounds = d * 3,
            distance = d,
            before_round_data_depolarization=noise
        )
        num_errors_sampled = count_logical_errors(circuit,num_shots)
        xs.append(noise)
        ys.append(num_errors_sampled/num_shots)
    plt.plot(xs,ys,label="d="+str(d))
plt.loglog()
plt.xlabel("p")
plt.ylabel("p_l")
plt.legend()
plt.show()

In [ ]:
tasks = [
    sinter.Task(
        circuit=stim.Circuit.generated(
            "repetition_code:memory",
            rounds=d*3,
            distance=d,
            before_round_data_depolarization=noise,
        ),
        json_metadata={"d":d,"p":noise}
    )
    for d in [3,5,7,9]
    for noise in [0.05,0.08,0.1,0.2,0.3,0.4,0.5]
]

collected_stats: List[sinter.TaskStats] = sinter.collect(
    num_workers=4,
    tasks=tasks,
    decoders=["pymatching"],
    max_shots=100_000,
    max_errors=500
)

In [ ]:
fig,ax=plt.subplots(1,1)
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats,
    x_func=lambda stats:stats.json_metadata["p"],
    group_func=lambda stats:stats.json_metadata["d"],
)
ax.set_ylim(1e-4,1e-0)
ax.set_xlim(5e-2,5e-1)
ax.loglog()
ax.set_title("title")
ax.set_xlabel("x")
ax.grid(which="major")
ax.grid(which="minor")
ax.legend()
fig.set_dpi(120)

In [ ]:
surface_code_circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=9,
    distance=3,
    after_clifford_depolarization=0.001,
    after_reset_flip_probability=0.001,
    before_measure_flip_probability=0.001,
    before_round_data_depolarization=0.001
)
surface_code_circuit.without_noise().diagram("timeslice-svg")

In [ ]:
surface_code_circuit.without_noise().diagram("detslice-svg")

In [ ]:
surface_code_tasks = [
    sinter.Task(
        circuit=stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            rounds=d * 3,
            distance=d,
            after_clifford_depolarization=noise,
            after_reset_flip_probability=noise,
            before_measure_flip_probability=noise,
            before_round_data_depolarization=noise
        ),
        json_metadata={"d":d,"r":d*3,"p":noise}
    )
    for d in [3,5,7]
    for noise in [.008,.009,.010,.011,.012]
]

In [ ]:
collected_surface_code_stats:List[sinter.TaskStats] = sinter.collect(
    num_workers=os.cpu_count() - 1,
    tasks=surface_code_tasks,
    decoders=['pymatching'],
    max_shots=1_000_000,
    max_errors=5_000,
    print_progress=True
)

In [ ]:
fig,ax = plt.subplots(1,1)
sinter.plot_error_rate(
    ax=ax,
    stats=collected_surface_code_stats,
    x_func=lambda stat: stat.json_metadata["p"],
    group_func=lambda stat:stat.json_metadata['d'],
    failure_units_per_shot_func=lambda stat:stat.json_metadata['r']
)
ax.set_ylim(5e-3,5e-2)
ax.set_xlim(0.008,0.012)
ax.loglog()
ax.legend()
fig.set_dpi(120)

In [8]:
assert alse

NameError: name 'alse' is not defined